<a href="https://colab.research.google.com/github/MaheyshRamesh/DataEngineering/blob/main/Copy_of_Assignment1_Data_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Data Profiling (EDA) Report
## Online Retail II: diagnosing and fixing data quality problems

| Member | Name | Matric number | Responsibility |
|---|---|---|---|
| Member 1 | Maheysh A/L Ramesh | A211881 | Dataset, setup, size/columns/dtypes, duplicates, data types, GitHub repo |
| Member 2 | Danish Irfan bin Hakimi | A213696 | _TODO_ |
| Member 3 | Shukri Aizat bin Shuazrin | A212202 | _TODO_ |
| Member 4 | Ahmad Nazmi bin Bastian Affendy | A212353 | _TODO_ |

**Contents:** 1 Scenario · 2 Data source · 3 Profiling and issue handling (3.1 load, 3.2 data types, 3.3 duplicates, 3.4 missing values, 3.5 outliers, 3.6 other issues) · 4 Diagnostic charts · 5 PII and bias risks · 6 Conclusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaheyshRamesh/DataEngineering/blob/main/Assignment1_Data_Profiling.ipynb)

> **How to run (Google Colab):** click the badge above (or upload this notebook to Colab), then choose *Runtime > Run all*. No installation or manual download is needed.
> **How to run (locally):** see `README.md`; install `requirements.txt`, then *Kernel > Restart & Run All*.
> The dataset is downloaded automatically on the first run (the Excel file is about 44 MB, so the first load takes a few minutes). Later runs in the same session use a cache.

In [ ]:
# Setup: install anything missing. This is a no-op on Google Colab, which already has these packages.
import importlib.util
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "openpyxl": "openpyxl", "pyarrow": "pyarrow", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

IN_COLAB = "google.colab" in sys.modules
print("Running on Google Colab" if IN_COLAB else "Running locally")

All required packages are already installed.
Running on Google Colab


In [ ]:
from pathlib import Path
import shutil
import time
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.dpi"] = 110

DATA_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
for d in (RAW_DIR, INTERIM_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

ZIP_PATH = RAW_DIR / "online_retail_ii.zip"
XLSX_PATH = RAW_DIR / "online_retail_II.xlsx"
RAW_CACHE = INTERIM_DIR / "raw_both_sheets.pkl"
print("Working directory:", ROOT)

Working directory: /content


## 1. Scenario

An online gift retailer wants to use its sales and customer data to improve business performance: it needs to identify its best-selling products, understand how customers buy, and predict future sales. All three analyses rely on clean transaction records, so before any modelling we must profile the data and repair its quality problems. Our questions are: (1) which products sell best by units and by revenue, (2) how do customers differ in how often and how much they buy, and (3) can weekly sales be forecast from the history?

**Why this dataset fits the brief** (real, messy, and has risks to discuss):
- It is a real transaction log, not a synthetic or pre-cleaned file, and it was never prepared for analysis.
- It contains missing values (customer IDs, descriptions), outliers (very large quantities and prices), duplicates, wrong or mixed data types, invalid values (negative quantities and prices), non-product rows, and cancellations.
- It has an identifier field (`Customer ID`) and a strong sampling bias (a single UK retailer, mostly UK buyers and many wholesalers), so PII and bias can be discussed.
- It spans two years (Dec 2009 to Dec 2011), enough to look at seasonality for the forecasting question.

## 2. Data source

| Item | Detail |
|---|---|
| Dataset | **Online Retail II** (UCI Machine Learning Repository, dataset id 502) |
| Page | https://archive.ics.uci.edu/dataset/502/online+retail+ii |
| Direct download | https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip |
| DOI | https://doi.org/10.24432/C5CG6D |
| Licence | Creative Commons Attribution 4.0 International (CC BY 4.0), as stated on the UCI page |
| Content | All transactions of a UK-based, registered non-store online retailer (mainly unique all-occasion gifts, many customers are wholesalers), 01/12/2009 to 09/12/2011 |
| How we obtained it | Downloaded programmatically from the UCI static URL above by the cell below. It is a public open-data download, so no scraping, login or API key is involved. The zip contains one Excel workbook, `online_retail_II.xlsx`, with two sheets (one per trading year). |

_Citation (as given on the UCI page):_ Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D.

In [ ]:
def download_dataset():
    """Download and extract the workbook once; skip if it is already on disk."""
    if XLSX_PATH.exists():
        print(f"Already downloaded: {XLSX_PATH.name} ({XLSX_PATH.stat().st_size / 1e6:.1f} MB)")
        return

    part_path = ZIP_PATH.with_suffix(".part")
    for attempt in range(1, 5):  # retry a few times: public servers can be slow or drop connections
        try:
            print(f"Downloading from UCI (attempt {attempt}) ...")
            request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0 (data-engineering-assignment)"})
            with urllib.request.urlopen(request, timeout=60) as response, open(part_path, "wb") as out:
                shutil.copyfileobj(response, out, length=1 << 20)  # stream to disk in 1 MB chunks
            if not zipfile.is_zipfile(part_path):
                raise ValueError("Downloaded file is not a valid zip")
            part_path.replace(ZIP_PATH)
            break
        except Exception as exc:
            print(f"  failed: {type(exc).__name__}: {exc}")
            if attempt == 4:
                raise RuntimeError(
                    "Could not download the dataset. Download the zip manually from "
                    f"{DATA_URL} and place online_retail_II.xlsx in {RAW_DIR}"
                ) from exc
            time.sleep(5 * attempt)

    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extract("online_retail_II.xlsx", RAW_DIR)
    print(f"Saved {XLSX_PATH.name} ({XLSX_PATH.stat().st_size / 1e6:.1f} MB)")

download_dataset()

Saved online_retail_II.xlsx (45.6 MB)


## 3. Data profiling and issue handling

### 3.1 Load the data and inspect size, columns and data types _(Member 1)_

The workbook has **two sheets** (one per trading year). We read both with the default pandas types, add a `source_sheet` column that records which sheet each row came from (this is our own lineage column, not part of the source data), and stack them. We keep this untouched copy as `raw` so every later fix can be compared against it. Reading Excel is slow, so the raw frame is cached as a pickle for later runs.

In [ ]:
if RAW_CACHE.exists():
    raw = pd.read_pickle(RAW_CACHE)
    print("Loaded cached raw data")
else:
    sheets = pd.read_excel(XLSX_PATH, sheet_name=None)  # dict: sheet name -> DataFrame
    frames = []
    for name, frame in sheets.items():
        frame = frame.copy()
        frame["source_sheet"] = name
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    raw.to_pickle(RAW_CACHE)
    print("Read Excel and cached raw data")

sheet_names = list(raw["source_sheet"].unique())
print(f"\nRows: {raw.shape[0]:,}   Columns: {raw.shape[1]}   Memory: {raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print("\nRows per sheet:")
print(raw.groupby("source_sheet").size().to_string())

In [ ]:
summary = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "missing": raw.isna().sum(),
    "n_unique": raw.nunique(dropna=True),
})
display(summary)
display(raw.head())

### 3.2 Incorrect data types: audit and fix _(Member 1)_

`DataFrame.dtypes` only reports `object` for text-like columns, which hides the real problem. So we look at the **Python type of each value**, and check whether each column's type matches what the column means.

In [ ]:
for col in ["Invoice", "StockCode", "Description"]:
    print(f"--- {col}: Python type of each value ---")
    print(raw[col].map(lambda v: type(v).__name__).value_counts(dropna=False).to_string())
    print()

cid = raw["Customer ID"].dropna()
print("Customer ID dtype:", raw["Customer ID"].dtype, "| all non-null values are whole numbers:", bool((cid % 1 == 0).all()))
print("InvoiceDate dtype:", raw["InvoiceDate"].dtype, "| range:", raw["InvoiceDate"].min(), "->", raw["InvoiceDate"].max())

prefix = raw["Invoice"].astype(str).str.extract(r"^([A-Za-z]*)")[0].replace("", "(digits only)")
print("\nInvoice prefixes (letters before the number):")
print(prefix.value_counts().to_string())

**Findings and decisions**

| Column | Problem found | Why it is wrong | Fix and reason |
|---|---|---|---|
| `Invoice` | Mixed Python types: Excel stored plain numbers as integers but numbers with a letter prefix (`C` for cancellations, `A` for adjustments) as text | An identifier is a label, not a quantity: you never add or average invoice numbers, and mixed types break joins, sorting and `groupby` | Convert to string. We keep the prefix, because the `C` prefix is the only marker for cancellations. |
| `StockCode` | Mixed integers and text (e.g. `85123A`, `POST`, `M`) | Same reason: product codes are identifiers | Convert to string |
| `Description` | Free text, but missing values are stored as float `NaN` and a few entries were read as integers (see the type counts above) | A text column should hold only text or an explicit missing marker | Convert to pandas `string` dtype, so numbers become text and missing values become `<NA>` |
| `Customer ID` | `float64` | Pandas stores it as float only because some values are missing (NaN), but an ID is a label. Float would also print as `13085.0` and could lose digits in joins | Convert float to nullable integer, then to string, so missing IDs stay missing instead of becoming a fake number |
| `Country` | `object`, few repeated labels | Low-cardinality label, wasteful as free text | Convert to `category` |
| `InvoiceDate` | Already `datetime64` | Correct, no fix needed | None |
| `Quantity`, `Price` | `int64` and `float64` | Correct for numeric measures | None (their invalid values are handled in 3.5 and 3.6) |

We also rename columns to `snake_case` (`Customer ID` has a space, and the names differ from the first UCI file), and add a boolean `is_cancellation` flag derived from the `C` prefix, so the information is not buried in a string.

In [ ]:
COLUMN_NAMES = {
    "Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "price",
    "Customer ID": "customer_id", "Country": "country",
}
ORIG_COLS = list(COLUMN_NAMES.values())  # the 8 source columns (excludes our own source_sheet)

df = raw.rename(columns=COLUMN_NAMES).copy()
dtypes_before = df.dtypes.astype(str)

for col in ["invoice", "stock_code", "description"]:
    df[col] = df[col].astype("string")

# Customer ID: float -> nullable integer -> string (missing stays missing)
assert (df["customer_id"].dropna() % 1 == 0).all(), "Customer IDs are not all whole numbers"
df["customer_id"] = df["customer_id"].astype("Int64").astype("string")

df["country"] = df["country"].astype("category")
df["is_cancellation"] = df["invoice"].str.startswith("C").astype(bool)

pd.DataFrame({"before": dtypes_before, "after": df.dtypes.astype(str)}).loc[list(df.columns)]

In [ ]:
# Proof that the conversion left consistent types
for col in ["invoice", "stock_code", "description", "customer_id"]:
    print(col, "->", df[col].map(lambda v: type(v).__name__).value_counts(dropna=False).to_dict())
print("\nExample IDs:", df["customer_id"].dropna().head(3).tolist(), "| missing IDs preserved:", int(df["customer_id"].isna().sum()))

### 3.3 Duplicates: identify and fix _(Member 1)_

**What counts as a duplicate here?** The data has no unique row ID, so a duplicate is a row that is identical in **all eight source columns** (invoice, stock code, description, quantity, timestamp to the minute, price, customer, country). We look at it in three steps, from clear-cut to ambiguous:

1. **Exact duplicates**, split into repeats *inside one sheet* and records that appear in *both sheets*.
2. **Hidden duplicates**, which only become exact after fixing trivial formatting (extra spaces, upper/lower case).
3. **Near-duplicates**, rows that match on everything except the description text.

In [ ]:
total_dups = int(df.duplicated(subset=ORIG_COLS).sum())
within = {s: int(g.duplicated(subset=ORIG_COLS).sum()) for s, g in df.groupby("source_sheet", observed=True)}
# distinct records that exist in both sheets = total duplicates minus repeats that happen inside a single sheet
cross_sheet = total_dups - sum(within.values())

dup_breakdown = pd.Series({**{f"repeated inside '{s}'": n for s, n in within.items()},
                           "record present in both sheets": cross_sheet})
print(f"Exact duplicate rows (extra copies): {total_dups:,}  ({total_dups / len(df):.2%} of all rows)\n")
print(dup_breakdown.to_string())

s1, s2 = sheet_names[0], sheet_names[1]
overlap_start = df.loc[df["source_sheet"] == s2, "invoice_date"].min()
overlap_end = df.loc[df["source_sheet"] == s1, "invoice_date"].max()
print(f"\nSheet '{s1}' ends at   {overlap_end}")
print(f"Sheet '{s2}' starts at {overlap_start}")
print("The two sheets overlap in time" if overlap_start < overlap_end else "The two sheets do not overlap in time")

In [ ]:
# Diagnostic chart 1: where do the duplicates come from?
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))

dup_breakdown.plot(kind="barh", ax=axes[0], color=["#4C78A8", "#4C78A8", "#E45756"])
axes[0].set_title("Exact duplicate rows by origin")
axes[0].set_xlabel("Number of extra copies")
for i, v in enumerate(dup_breakdown.values):
    axes[0].text(v, i, f" {v:,}", va="center")
axes[0].set_xlim(0, dup_breakdown.max() * 1.18)

lo, hi = overlap_start.normalize() - pd.DateOffset(days=10), overlap_end.normalize() + pd.DateOffset(days=10)
window = df[(df["invoice_date"] >= lo) & (df["invoice_date"] < hi)]
daily = (window.groupby([window["invoice_date"].dt.normalize(), "source_sheet"], observed=True)
               .size().unstack(fill_value=0))
daily.plot(kind="bar", ax=axes[1], width=0.85, color=["#4C78A8", "#F58518"])
axes[1].set_xticklabels([d.strftime("%d %b") for d in daily.index], rotation=90)
axes[1].set_title("Rows per day around the sheet overlap")
axes[1].set_xlabel("Invoice date")
axes[1].set_ylabel("Rows")
axes[1].legend(title="Source sheet")

plt.tight_layout()
plt.show()

In [ ]:
# What do exact duplicates look like? (a few examples, grouped so copies sit together)
dup_rows = df[df.duplicated(subset=ORIG_COLS, keep=False)].sort_values(["invoice", "stock_code", "invoice_date"])
display(dup_rows[ORIG_COLS + ["source_sheet"]].head(8))

print("Invoices containing at least one exact-duplicate line:", dup_rows["invoice"].nunique())

**Decision for exact duplicates: remove all extra copies (keep the first occurrence).**

- *Cross-sheet records* are clearly an artefact of how the workbook was split: the first sheet runs a few days into December 2010 and the second sheet starts at 1 December 2010, so those days appear twice. Keeping them would **double-count sales** for those days, which directly distorts the best-seller ranking and the weekly forecast.
- *Repeats inside one sheet* are less certain. A shop can legitimately scan the same product twice, and no row ID lets us prove otherwise. However, rows that match on invoice, product, quantity, price, customer **and the same minute** are far more likely to be accidental double entries than genuine separate lines, and a genuine repeat would normally appear as one line with a larger quantity. We therefore remove them as well, and **measure the revenue impact below** so the effect of this judgement is visible. This remains a stated limitation.

In [ ]:
# Step 2: hidden duplicates. Normalise trivial formatting differences, then recount.
df["stock_code"] = df["stock_code"].str.strip().str.upper()
df["description"] = df["description"].str.strip().str.replace(r"\s+", " ", regex=True)

total_dups_after_norm = int(df.duplicated(subset=ORIG_COLS).sum())
print(f"Exact duplicates before text normalisation: {total_dups:,}")
print(f"Exact duplicates after  text normalisation: {total_dups_after_norm:,}")
print(f"Hidden duplicates revealed by normalisation: {total_dups_after_norm - total_dups:,}")

raw_desc = raw["Description"].dropna().astype(str)
raw_codes = raw["StockCode"].astype(str)
print(f"\nDescriptions with leading/trailing spaces in the raw data: {int((raw_desc != raw_desc.str.strip()).sum()):,}")
print(f"Distinct stock codes lost by trimming/upper-casing (same product written two ways): {raw_codes.nunique() - raw_codes.str.strip().str.upper().nunique():,}")

Text normalisation (strip spaces, collapse repeated spaces, upper-case stock codes) is safe here: `85123a` and `85123A` are the same product, and padded spaces carry no meaning. The recount above shows whether this exposes any further exact duplicates.

In [ ]:
# Step 3: near-duplicates = identical on everything except the description text.
# Evaluated after exact duplicates are removed, so we only look at what is left.
deduped = df.drop_duplicates(subset=ORIG_COLS, keep="first")
KEY_NO_DESC = [c for c in ORIG_COLS if c != "description"]
near_mask = deduped.duplicated(subset=KEY_NO_DESC, keep=False)
print("Rows that match another row on all fields except description:", int(near_mask.sum()))
display(deduped.loc[near_mask, ORIG_COLS].sort_values(["invoice", "stock_code"]).head(12))

**Decision for near-duplicates: flag and keep (do not delete).** These rows agree on invoice, product code, quantity, timestamp, price and customer, and differ only in how the product is described. They may be one line entered twice with a text variant, but they could also be two distinct lines on the same invoice. Because the evidence is weaker than for exact duplicates and the count is tiny, deleting them would be a guess. We add an `is_near_duplicate` flag instead so later members can test how sensitive their results are.

In [ ]:
# Apply the fixes
n_before = len(df)
df["line_total"] = df["quantity"] * df["price"]
removed = df[df.duplicated(subset=ORIG_COLS, keep="first")]

clean = df.drop_duplicates(subset=ORIG_COLS, keep="first").reset_index(drop=True)
clean["is_near_duplicate"] = clean.duplicated(subset=KEY_NO_DESC, keep=False)

# Revenue impact of what was removed (line total = quantity x price)
impact = pd.DataFrame({
    "rows": [n_before, len(clean), len(removed)],
    "net_line_total": [df["line_total"].sum(), clean["line_total"].sum(), removed["line_total"].sum()],
}, index=["before", "after", "removed"])
impact["share_of_before"] = impact["net_line_total"] / impact.loc["before", "net_line_total"]
display(impact)

print(f"Removed {len(removed):,} rows ({len(removed) / n_before:.2%}), which carried {removed['line_total'].sum() / df['line_total'].sum():.2%} of net line-total value.")
print(f"Flagged near-duplicate rows kept: {int(clean['is_near_duplicate'].sum())}")

In [ ]:
# Validation: the fixes must hold, and the numbers must reconcile
assert not clean.duplicated(subset=ORIG_COLS).any(), "exact duplicates remain"
assert len(clean) == n_before - len(removed), "row counts do not reconcile"
assert str(clean["customer_id"].dtype) == "string" and str(clean["invoice"].dtype) == "string"
assert str(clean["stock_code"].dtype) == "string" and str(clean["country"].dtype) == "category"
assert pd.api.types.is_datetime64_any_dtype(clean["invoice_date"])
assert clean["customer_id"].isna().sum() == raw["Customer ID"].isna().sum() - removed["customer_id"].isna().sum()

OUT_PATH = PROCESSED_DIR / "online_retail_ii_step1.parquet"
clean.to_parquet(OUT_PATH, index=False)
print("All checks passed. Saved:", OUT_PATH.name, "| shape:", clean.shape)

### 3.3 summary: what Member 1 changed

| Step | Issue | Action | Reason |
|---|---|---|---|
| Types | Mixed int/str identifiers, `Customer ID` as float | Converted to `string` (IDs keep missing values) | Identifiers are labels, mixed types break joins and grouping |
| Types | No cancellation indicator | Added boolean `is_cancellation` from the `C` prefix | Needed by later steps, and removes string-parsing everywhere |
| Names | Inconsistent names with spaces | `snake_case` | Safer code, consistent across the two UCI files |
| Duplicates | Records present in both sheets, and repeats inside one sheet | Dropped, keeping first occurrence | Overlapping days would double-count sales |
| Duplicates | Spacing/case variants | Normalised `stock_code` and `description`, then recounted | Checks for duplicates hidden by formatting. The recount found none extra, but normalisation still merges the same product written two different ways (e.g. `85123a` vs `85123A`), which matters for best-seller counts |
| Duplicates | Same line, different description text | Flagged (`is_near_duplicate`), kept | Evidence is too weak to delete |

**Handed to the next steps (found while profiling, not fixed here):** the counts below are the starting point for missing values, outliers and invalid values.

In [ ]:
handoff = pd.Series({
    "missing customer_id": int(clean["customer_id"].isna().sum()),
    "missing description": int(clean["description"].isna().sum()),
    "cancellation rows (invoice starts with C)": int(clean["is_cancellation"].sum()),
    "quantity <= 0 on non-cancellation rows": int(((clean["quantity"] <= 0) & ~clean["is_cancellation"]).sum()),
    "price <= 0": int((clean["price"] <= 0).sum()),
    "stock codes with no digit (POST, DOT, M, BANK CHARGES ...)": int((~clean["stock_code"].str.contains(r"\d", na=False)).sum()),
    "country = 'Unspecified' or 'European Community'": int(clean["country"].isin(["Unspecified", "European Community"]).sum()),
    "stock codes with more than one description": int((clean.groupby("stock_code")["description"].nunique() > 1).sum()),
})
display(handoff.rename("rows / codes").to_frame())
print("Quantity range:", clean["quantity"].min(), "to", clean["quantity"].max(), "| Price range:", clean["price"].min(), "to", clean["price"].max())

### 3.4 Missing values _(TODO: next member)_
_Where and how much is missing, how it is handled (drop / impute / flag) and why. Start from the table above (`customer_id`, `description`)._

### 3.5 Outliers _(TODO)_
_Detection method (IQR / z-score / boxplot), and whether values are kept, capped or removed, with reasons. Quantity and price have extreme values at both ends._

### 3.6 Other quality issues _(TODO)_
_Invalid values (negative or zero quantity/price, non-product stock codes), inconsistent formats and categories (country labels, description variants per stock code), and cancellations._

## 4. Exploratory (diagnostic) charts
_At least 2 labelled charts in total. Chart 1 (duplicate origins and sheet overlap) is already in section 3.3. Add missing-value and outlier charts here._

## 5. PII and bias risks _(TODO)_
_Sensitive or identifying fields (`customer_id`), sampling bias (single UK retailer, mostly UK, many wholesalers, customers without an ID), unbalanced groups, and how each would be handled._

## 6. Conclusion _(TODO)_
_Whether the data is fit for the scenario and which limitations remain._


### 3.4 Missing Values and Inconsistent Formats (Member 2)

This section identifies missing values, handles them using suitable methods, and checks inconsistent text and date formats. The cleaning decisions are made based on the purpose of each column.


In [ ]:

# 3.4.1 Missing Value Detection

missing_before = pd.DataFrame({
    "Missing Count": clean.isnull().sum(),
    "Missing Percentage (%)": (
        clean.isnull().sum() / len(clean) * 100
    ).round(2)
})

print("Missing Values Before Cleaning:")
display(missing_before)



### 3.4.2 Missing Value Visualization

A bar chart is used to show the percentage of missing values in each column. This helps us identify which columns have the most missing data and need further cleaning.


In [ ]:

# 3.4.2 Missing Value Bar Chart

missing_percent = (
    clean.isnull().mean() * 100
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))

bars = plt.bar(
    missing_percent.index,
    missing_percent.values,
    color="steelblue"
)

plt.title("Missing Values Before Cleaning")
plt.xlabel("Columns")
plt.ylabel("Missing Values (%)")
plt.xticks(rotation=45, ha="right")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.2,
        f"{height:.2f}%",
        ha="center"
    )

plt.ylim(0, max(missing_percent.max() * 1.15, 1))
plt.tight_layout()
plt.show()



### 3.4.3 Handling Missing Values

Different methods are used to handle missing values based on the meaning of each column.

- **Description:** Missing descriptions are replaced with "UNKNOWN PRODUCT" because the transaction may still contain useful information. This avoids removing valid transactions.
- **Customer ID:** Missing customer IDs are kept and flagged because filling them with random or common IDs could give incorrect customer information.
- **Other important columns:** Rows with missing essential transaction fields are removed only if such missing values are found, because these fields are needed for reliable transaction analysis.



In [ ]:

# 3.4.3 Handle Missing Values

# Copy Member 1's cleaned dataset
clean_m2 = clean.copy()

# Flag missing Customer IDs
clean_m2["customer_id_missing"] = (
    clean_m2["customer_id"].isnull()
)

# Fill missing product descriptions
clean_m2["description"] = (
    clean_m2["description"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("UNKNOWN PRODUCT")
)

# Check essential transaction columns
essential_columns = [
    "invoice", "stock_code", "quantity",
    "invoice_date", "price", "country"
]

rows_before = len(clean_m2)

clean_m2 = clean_m2.dropna(
    subset=essential_columns
).copy()

rows_dropped = rows_before - len(clean_m2)

print("Rows removed due to missing essential values:", rows_dropped)
print("Missing Customer IDs retained:",
      clean_m2["customer_id"].isnull().sum())

print("\nMissing Values After Handling:")
display(clean_m2.isnull().sum().to_frame("Missing Count"))



### 3.4.4 Standardizing Date and Text Formats

Inconsistent formats can affect data analysis because the same information may be represented differently. We standardize product descriptions and country text by removing unnecessary spaces. We also verify that invoice dates use a consistent datetime data type.

Some text formatting was already handled by Member 1, so this step continues the existing cleaning process.


In [ ]:

# 3.4.4 Standardize Text and Date Formats

# Standardize product descriptions
clean_m2["description"] = (
    clean_m2["description"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)

# Standardize country text
clean_m2["country"] = (
    clean_m2["country"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)
# Convert country back to category data type
clean_m2["country"] = clean_m2["country"].astype("category")

# Ensure InvoiceDate uses datetime format
clean_m2["invoice_date"] = pd.to_datetime(
    clean_m2["invoice_date"],
    errors="coerce"
)

print("Data Types After Standardization:")
display(clean_m2[
    ["description", "country", "invoice_date"]
].dtypes.to_frame("Data Type"))

print("\nSample Data After Standardization:")
display(clean_m2[
    ["description", "country", "invoice_date"]
].head(10))


In [ ]:

# Check text formatting changes (Before vs After)

# Compare product descriptions
description_before = clean["description"].fillna("UNKNOWN PRODUCT").astype("string")

description_after = clean_m2["description"].astype("string")

description_changed = (
    description_before.ne(description_after).fillna(False).sum()
)

# Compare country text
country_before = clean["country"].astype("string")
country_after = clean_m2["country"].astype("string")

country_changed = (
    country_before.ne(country_after).fillna(False).sum()
)

# Display results
print("Text Formatting Changes:")
print("Description values changed:", description_changed)
print("Country values changed:", country_changed)
print("Invoice Date Type:", clean_m2["invoice_date"].dtype)



### 3.4.5 Missing Values Before and After Cleaning

The table below compares missing values before and after the cleaning process. This allows us to evaluate whether the selected handling methods were effective while keeping useful transaction information.


In [ ]:

# 3.4.5 Before vs After Comparison

comparison = pd.DataFrame({
    "Before Missing": clean.isnull().sum(),
    "After Missing": clean_m2[clean.columns].isnull().sum(),
    "Before (%)": (
        clean.isnull().mean() * 100
    ).round(2),
    "After (%)": (
        clean_m2[clean.columns].isnull().mean() * 100
    ).round(2)
})

print("Missing Values: Before vs After")
display(comparison)

print("\nOriginal Rows:", len(clean))
print("Rows After Cleaning:", len(clean_m2))
print("Rows Removed:", len(clean) - len(clean_m2))



### 3.4.6 Discussion and Justification

After checking the dataset, we found that some columns contain missing values, especially customer IDs and product descriptions.

For missing product descriptions, we replaced the empty values with "UNKNOWN PRODUCT" because the other transaction information is still useful for analysis. We decided not to remove these rows to avoid losing unnecessary data.

For missing customer IDs, we kept the missing values and added a flag column. This is because customer IDs are unique identifiers, and replacing them with estimated values could lead to incorrect customer analysis.

We also checked inconsistent date and text formats. Product descriptions were standardized by removing extra spaces and converting them to uppercase. Country names were cleaned by removing unnecessary spaces, while invoice dates were converted into a consistent datetime format.

Overall, the cleaning process improved data consistency while preserving useful transaction records. However, missing customer IDs still remain and should be considered when performing customer-level analysis.


### 4. Outlier Detection and Handling

In this section, we will identify outliers and invalid values in the Online Retail II dataset. We will focus on the quantity and price columns because unusual values in these columns can affect sales analysis. We will use the IQR method to detect potential outliers and decide how to handle them based on the transaction records.

In [ ]:
#Check the data before outlier detection

# Create a copy so the original cleaned data is not changed
df_member3 = clean.copy()

# Check the dataset size
print("Dataset shape:", df_member3.shape)

# Display summary statistics for quantity and price
display(df_member3[['quantity', 'price']].describe())


#### 4.1 Detecting Outliers Using IQR

We used the Interquartile Range (IQR) method to identify outliers in the quantity and price columns. The IQR method helps us detect values that are unusually high or low compared to most transactions. We will examine these outliers before deciding whether to keep or remove them, because some unusual values may still represent valid sales.

In [ ]:

# Detect outliers using the IQR method

for col in ['quantity', 'price']:
    Q1 = df_member3[col].quantile(0.25)
    Q3 = df_member3[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_member3[
        (df_member3[col] < lower_bound) |
        (df_member3[col] > upper_bound)
    ]

    print(f"\nColumn: {col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Number of outliers:", len(outliers))
    print("Percentage of outliers:",
          round(len(outliers) / len(df_member3) * 100, 2), "%")


### 3.6.2 Category consistency and unresolved labels (Member 4)

**Input and scope.** We start from Member 2's `clean_m2`, which includes Member 1's
deduplication and type corrections. We create `clean_m4` without changing either
friend's DataFrame. Member 2 already standardized descriptions and country text;
we audit the result rather than claiming the same corrections again.

The rules below remove only remaining spacing differences and merge country labels
that differ solely by letter case. The most frequent normalized spelling is retained
(alphabetical order breaks ties). This is a formatting rule, not a verification of
a customer's actual location. Distinct labels such as `EIRE` are retained without an
externally verified location dictionary. `Unspecified` and `European Community`
do not identify a specific country, so we retain and flag them rather than guess.

A product code with several known descriptions is **a review candidate**, not a
proven error: wording can change over time. We exclude the `UNKNOWN PRODUCT`
placeholder from the count of known descriptions, retain the supplied description,
and flag the affected codes. No modal description is copied over another value.

In [ ]:
from IPython.display import Markdown, display

if 'clean_m2' not in globals():
    raise RuntimeError('Run the Member 1 and Member 2 cells above before Member 4.')

clean_m4 = clean_m2.copy(deep=True)
m4_input_rows = len(clean_m2)
if m4_input_rows == 0:
    raise ValueError('Member 4 requires a non-empty input dataset.')

country_before_m4 = clean_m2['country'].astype('string')
country_trimmed = country_before_m4.str.strip().str.replace(r'\s+', ' ', regex=True)
country_trimmed = country_trimmed.replace('', pd.NA)
country_spellings = country_trimmed.value_counts().rename_axis('label').reset_index(name='rows')
country_spellings['case_key'] = country_spellings['label'].str.casefold()
preferred_spelling = (
    country_spellings.sort_values(['rows', 'label'], ascending=[False, True])
    .drop_duplicates('case_key').set_index('case_key')['label']
)
country_after_m4 = country_trimmed.str.casefold().map(preferred_spelling).astype('string')
clean_m4['country'] = country_after_m4.astype('category')

country_changed_m4 = country_before_m4.fillna('<MISSING>').ne(country_after_m4.fillna('<MISSING>'))
ambiguous_location = country_after_m4.str.casefold().isin(['unspecified', 'european community'])
clean_m4['country_label_unresolved'] = (country_after_m4.isna() | ambiguous_location).astype(bool)

desc_unknown_m4 = (
    clean_m4['description'].isna()
    | clean_m4['description'].str.strip().eq('').fillna(False)
    | clean_m4['description'].eq('UNKNOWN PRODUCT').fillna(False)
)
known_descriptions_per_code_m4 = (
    clean_m4.loc[~desc_unknown_m4].groupby('stock_code', observed=True)['description'].nunique()
)
clean_m4['description_unresolved'] = desc_unknown_m4.astype(bool)
clean_m4['product_description_review'] = (
    clean_m4['stock_code'].map(known_descriptions_per_code_m4).fillna(0).gt(1).astype(bool)
)

category_audit_m4 = pd.DataFrame({
    'measure': [
        'Input transaction lines', 'Country labels before', 'Country labels after',
        'Country values changed by Member 4', 'Lines with unresolved location labels',
        'Lines with unresolved descriptions', 'Stock codes with multiple known descriptions',
        'Lines belonging to those stock codes', 'Transaction lines deleted by Member 4'
    ],
    'count': [
        m4_input_rows, country_before_m4.nunique(), country_after_m4.nunique(),
        int(country_changed_m4.sum()), int(clean_m4['country_label_unresolved'].sum()),
        int(desc_unknown_m4.sum()), int(known_descriptions_per_code_m4.gt(1).sum()),
        int(clean_m4['product_description_review'].sum()), 0
    ]
})
display(category_audit_m4)
display(Markdown(
    f"**Finding:** {int(country_changed_m4.sum()):,} country values required additional "
    'formatting changes after Member 2. Zero changes means the audit confirmed '
    'existing consistency; it does not mean the category work was skipped.'
))

country_label_inventory_m4 = country_after_m4.value_counts(dropna=False).rename_axis('country_label').reset_index(name='transaction_lines')
display(country_label_inventory_m4)
description_review_summary_m4 = (
    known_descriptions_per_code_m4[known_descriptions_per_code_m4.gt(1)]
    .sort_values(ascending=False).rename('known_description_count').to_frame()
)
display(description_review_summary_m4.head(10))

**Category decisions and limits.** Spelling checks cannot prove factual accuracy.
An unresolved location stays in the dataset and remains visible in group totals.
A description review flag applies to all rows of the affected product code; its
row count is not a count of confirmed incorrect descriptions. Product grouping
should use `stock_code` while unresolved wording is reviewed against a product
catalogue, if available. No catalogue was supplied, so no semantic replacements
are asserted. The known missing-description placeholder is flagged, not described
as a recovered product name.

## 4. Exploratory diagnostic charts

Member 1's duplicate-origin chart is in section 3.3 and Member 2's missing-value
chart is in section 3.4. Member 4 adds geographic representation and customer-ID
coverage charts in section 5.2. All are data-quality diagnostics. These charts do
not answer the business's best-seller or forecasting questions. Member 3's outlier
diagnostic still needs to be inserted in section 3.5.

## 5. PII and bias risks (Member 4)

### 5.1 Privacy inventory and practical safeguards

The source schema has no explicit name, email, postal address or card-number
column. That observation does **not** establish anonymity. `customer_id` is a
persistent pseudonymous identifier: it links transactions and could link to an
external customer register. Invoice identifiers, precise timestamps, location and
purchase history can also help link records when combined.

The following inventory separates observed fields from proposed controls for a
real company. We do not claim to have anonymized the underlying public dataset.

In [ ]:
privacy_inventory_m4 = pd.DataFrame([
    {
        'field(s)': 'customer_id',
        'risk': 'Persistent pseudonymous identifier; links a customer across purchases.',
        'handling': 'Retain only in restricted customer-level analysis; omit from shared diagnostics. Do not infer missing IDs.'
    },
    {
        'field(s)': 'invoice, invoice_date',
        'risk': 'Transaction key and precise time may support linkage to a receipt or another system.',
        'handling': 'Publish aggregate periods instead of invoice-level examples; restrict detailed records.'
    },
    {
        'field(s)': 'country',
        'risk': 'A broad location is low detail alone but may increase linkage risk in rare combinations.',
        'handling': 'Use aggregate group counts; review small groups before external release.'
    },
    {
        'field(s)': 'stock_code, description, quantity, price, line_total',
        'risk': 'Purchase history and spending may be sensitive business/customer information when linked.',
        'handling': 'Share only necessary aggregate summaries; keep transaction-level exports controlled.'
    },
    {
        'field(s)': 'Names, addresses, emails, payment-card numbers',
        'risk': 'No explicit fields of these types exist in the supplied schema; free text is not exhaustively verified.',
        'handling': 'For a live retailer, inspect free text and collection practices before release; apply data minimization.'
    }
])
display(privacy_inventory_m4)
print('Member 4 outputs below contain aggregate diagnostics, not customer or invoice examples.')

**Implemented here:** new charts and tables summarize groups without displaying
customer identifiers or invoice identifiers. The working DataFrame retains its
IDs because they are needed for legitimate customer-level analysis, and a local
working Parquet file is regenerated under the repository's ignored `data/` folder.
This is not an anonymized public export. Earlier members' original example rows
are preserved in their sections; they still contain public UCI identifiers.

**For a live company's data:** restrict access to detailed records, minimize stored
fields, use appropriate retention controls, and apply an agreed small-group release
rule before publishing aggregates. If customer linkage must be shared, use controlled
tokenization with any lookup kept separately. A hash of a short numeric ID is not
a guarantee of anonymity, and dropping an ID alone does not remove linkage risk.

### 5.2 Representation and customer-ID coverage

We measure transaction-line shares, not shares of people. One invoice can have
several lines and one customer can buy repeatedly. A dominant location therefore
shows the geographic concentration of the recorded activity; it does not by
itself prove demographic discrimination or establish the retailer's customer mix.
Unknown-ID rows remain in the denominator, so their absence does not disappear
from the quality report.

In [ ]:
country_bias_m4 = (
    clean_m4.assign(country_label=clean_m4['country'].astype('string').fillna('(MISSING)'))
    .groupby('country_label', observed=True)
    .agg(transaction_lines=('invoice', 'size'),
         missing_customer_lines=('customer_id_missing', 'sum'))
    .sort_values('transaction_lines', ascending=False)
)
country_bias_m4['line_share_pct'] = country_bias_m4['transaction_lines'] / len(clean_m4) * 100
country_bias_m4['missing_customer_pct'] = (
    country_bias_m4['missing_customer_lines'] / country_bias_m4['transaction_lines'] * 100
)
display(country_bias_m4)

coverage_m4 = pd.DataFrame({
    'customer_id_status': np.where(clean_m4['customer_id'].isna(), 'ID missing', 'ID recorded'),
    'is_uk': clean_m4['country'].astype('string').eq('United Kingdom').fillna(False).to_numpy(dtype=bool),
    'is_cancellation': clean_m4['is_cancellation'].to_numpy(dtype=bool)
})
id_coverage_m4 = coverage_m4.groupby('customer_id_status').agg(
    transaction_lines=('is_uk', 'size'),
    uk_line_share_pct=('is_uk', lambda x: x.mean() * 100),
    cancellation_line_share_pct=('is_cancellation', lambda x: x.mean() * 100)
)
id_coverage_m4['line_share_pct'] = id_coverage_m4['transaction_lines'] / len(clean_m4) * 100
display(id_coverage_m4)

def bias_stage_row(label, frame, country_col, customer_col):
    return {
        'stage': label, 'transaction_lines': len(frame),
        'missing_customer_pct': frame[customer_col].isna().mean() * 100,
        'uk_line_share_pct': frame[country_col].astype('string').eq('United Kingdom').fillna(False).mean() * 100
    }

bias_stages_m4 = pd.DataFrame([
    bias_stage_row('Raw workbook', raw, 'Country', 'Customer ID'),
    bias_stage_row('After Member 1', clean, 'country', 'customer_id'),
    bias_stage_row('After Member 2', clean_m2, 'country', 'customer_id'),
    bias_stage_row('After Member 4', clean_m4, 'country', 'customer_id')
])
display(bias_stages_m4)

uk_share_m4 = float(country_bias_m4.loc['United Kingdom', 'line_share_pct'])
missing_id_count_m4 = int(clean_m4['customer_id'].isna().sum())
missing_id_pct_m4 = missing_id_count_m4 / len(clean_m4) * 100
display(Markdown(
    f'**Measured risks:** United Kingdom-labelled rows account for **{uk_share_m4:.2f}%** '
    f'of transaction lines. **{missing_id_count_m4:,} lines ({missing_id_pct_m4:.2f}%)** '
    'cannot be assigned to an identified customer. Removing those rows would change '
    'the population available for analysis. The ID-status table shows observable '
    'differences in recorded location and cancellation coverage; it cannot establish '
    'why IDs are missing or prove that missingness is random.'
))

In [ ]:
# Diagnostic chart M4-A: geographic representation and identification coverage.
top_country_m4 = country_bias_m4.head(8)
location_shares_m4 = top_country_m4['line_share_pct'].copy()
remaining_share_m4 = country_bias_m4['line_share_pct'].iloc[8:].sum()
if remaining_share_m4 > 0:
    location_shares_m4.loc['All other location labels'] = remaining_share_m4

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
location_shares_m4.iloc[::-1].plot.barh(ax=axes[0], color='#276782')
axes[0].set(title='Geographic concentration of transaction lines',
            xlabel='Share of all transaction lines (%)', ylabel='Recorded location label')
axes[0].set_xlim(0, 100)
for i, value in enumerate(location_shares_m4.iloc[::-1]):
    axes[0].text(value + 0.6, i, f'{value:.2f}%', va='center', fontsize=8)

top_country_m4['missing_customer_pct'].iloc[::-1].plot.barh(ax=axes[1], color='#b46a37')
axes[1].set(title='Missing customer IDs in the eight largest locations',
            xlabel='Lines missing a customer ID within each location (%)', ylabel='')
axes[1].set_xlim(0, 100)
for i, value in enumerate(top_country_m4['missing_customer_pct'].iloc[::-1]):
    axes[1].text(value + 0.6, i, f'{value:.2f}%', va='center', fontsize=8)
fig.suptitle(f'Data-quality diagnostic | After Members 1, 2 and 4 | n = {len(clean_m4):,} transaction lines', fontsize=11)
fig.tight_layout()
plt.show()

In [ ]:
month_key_m4 = clean_m4['invoice_date'].dt.to_period('M').astype('string')
monthly_bias_m4 = (
    clean_m4.assign(calendar_month=month_key_m4)
    .groupby('calendar_month', observed=True)
    .agg(transaction_lines=('invoice', 'size'),
         missing_customer_lines=('customer_id_missing', 'sum'),
         first_timestamp=('invoice_date', 'min'), last_timestamp=('invoice_date', 'max'))
)
monthly_bias_m4['missing_customer_pct'] = (
    monthly_bias_m4['missing_customer_lines'] / monthly_bias_m4['transaction_lines'] * 100
)
display(monthly_bias_m4)

fig, ax = plt.subplots(figsize=(12, 4.6))
positions_m4 = np.arange(len(monthly_bias_m4))
ax.plot(positions_m4, monthly_bias_m4['missing_customer_pct'], marker='o', color='#276782', linewidth=2)
ax.set_xticks(positions_m4, monthly_bias_m4.index, rotation=55, ha='right')
ax.set(title='Customer-ID missingness varies across recorded calendar months',
       xlabel='Calendar month (final month is partial)',
       ylabel='Transaction lines missing a customer ID (%)', ylim=(0, 100))
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

first_date_m4 = clean_m4['invoice_date'].min()
last_date_m4 = clean_m4['invoice_date'].max()
display(Markdown(
    f'**Observed time coverage:** {first_date_m4:%d %B %Y} to {last_date_m4:%d %B %Y}. '
    'December 2011 ends on 9 December and must not be treated as a complete month '
    'when comparing monthly totals. A time range alone cannot prove that every '
    'transaction was captured. Missing-ID differences over time indicate uneven '
    'coverage and do not reveal their cause.'
))

### 5.3 Bias interpretation and mitigation

| Risk | Evidence or limit | Appropriate response |
|---|---|---|
| Geographic concentration | Country line shares and country-specific missing-ID rates above | Describe results as this retailer's recorded activity; report location-specific coverage and uncertainty. Do not duplicate underrepresented lines to balance sales totals. |
| Customer identification selection | Missing-ID share and differences between ID-status groups above | Retain unidentified transactions for appropriate transaction-level work; limit customer-level conclusions to recorded IDs and disclose the excluded share. Do not assign all unknown buyers one customer ID. |
| Source and sector sampling | UCI describes one UK gift retailer with many wholesalers | Avoid claims about all e-commerce or retail consumers. Large baskets are not automatically errors. The schema has no reliable wholesale/retail customer-type field. |
| Historical and time coverage | Transactions are from 2009-2011; the last month is partial | Treat any future forecast as a historical exercise, use time-ordered evaluation after cleaning, and seek current data for present-day decisions. Do not interpret partial-month totals as a sales decline. |
| Cleaning-related selection | Stage comparison reports missing-ID and UK shares before and after existing cleaning | Keep before/after group profiles; document which rows any future rule removes. Member 4 deletes no lines and imputes no IDs. |
| Unmeasured demographic fairness | No age, gender or other protected demographic fields are provided | Do not claim demographic fairness or discrimination has been measured. Location is not a substitute for these attributes. |

Geographic and temporal imbalance do not have a universal numerical 'fix'.
Coverage and limitations must be reported in relation to the proposed use.
No business insights, customer segmentation or forecasting model are produced
in this assignment section.

### 5.4 Validation and handoff

We validate that category checks and review flags did not alter upstream numeric
values, identifiers, descriptions, dates, missing-ID handling or row order. The
only permitted upstream-field change is justified country formatting. Added flags
record uncertainty; they do not resolve it. Country and monthly summaries must
account for every input line.

In [ ]:
assert len(clean_m4) == len(clean_m2)
assert clean_m4.index.equals(clean_m2.index)
protected_cols_m4 = [c for c in clean_m2.columns if c != 'country']
pd.testing.assert_frame_equal(clean_m4[protected_cols_m4], clean_m2[protected_cols_m4])
assert clean_m4['customer_id_missing'].equals(clean_m4['customer_id'].isna())
for flag in ['country_label_unresolved', 'description_unresolved', 'product_description_review']:
    assert clean_m4[flag].dtype == bool and not clean_m4[flag].isna().any()
assert int(country_bias_m4['transaction_lines'].sum()) == len(clean_m4)
assert np.isclose(country_bias_m4['line_share_pct'].sum(), 100)
assert int(monthly_bias_m4['transaction_lines'].sum()) == len(clean_m4)
assert int(country_bias_m4['missing_customer_lines'].sum()) == missing_id_count_m4
assert int(monthly_bias_m4['missing_customer_lines'].sum()) == missing_id_count_m4
assert clean_m4['country'].astype('string').equals(country_after_m4)

# Regenerated local working outputs. The repository already ignores data/.
M4_DIR = PROCESSED_DIR / 'member4_diagnostics'
M4_DIR.mkdir(parents=True, exist_ok=True)
M4_PATH = PROCESSED_DIR / 'online_retail_ii_member4_after_step2.parquet'
clean_m4.to_parquet(M4_PATH, index=False)
for filename, frame, include_index in [
    ('category_audit.csv', category_audit_m4, False),
    ('country_label_inventory.csv', country_label_inventory_m4, False),
    ('country_bias.csv', country_bias_m4, True),
    ('customer_id_coverage.csv', id_coverage_m4, True),
    ('monthly_bias.csv', monthly_bias_m4, True),
    ('bias_by_cleaning_stage.csv', bias_stages_m4, False),
    ('privacy_inventory.csv', privacy_inventory_m4, False),
]:
    frame.to_csv(M4_DIR / filename, index=include_index)
print('Member 4 validation passed: row order/count and all protected upstream fields are unchanged.')
print('Local working dataset:', M4_PATH.name, '| shape:', clean_m4.shape)
print('This file is after Member 2 plus Member 4; it is not a Member 3-completed dataset.')

## 6. Conclusion (Member 4)

The dataset is suitable for demonstrating retail data profiling, and the completed
stages have made its structure and limitations clearer. It is **not yet ready for
unrestricted downstream sales or forecasting analysis**: Member 3's outlier,
invalid-value and transaction-rule decisions have not been supplied. No model or
business performance claim has been made here.

In [ ]:
conclusion_m4 = f"""
**Measured state after Members 1, 2 and 4:** {len(clean_m4):,} transaction lines are retained.
Member 4 changed {int(country_changed_m4.sum()):,} country values and removed no lines.
There are {missing_id_count_m4:,} lines ({missing_id_pct_m4:.2f}%) without customer IDs,
{int(clean_m4['description_unresolved'].sum()):,} unresolved descriptions,
{int(clean_m4['country_label_unresolved'].sum()):,} unresolved location labels, and
{int(known_descriptions_per_code_m4.gt(1).sum()):,} stock codes with more than one known description.
These review counts are not all confirmed errors.

- **Best-selling products:** conditionally suitable once the group agrees how returns,
  cancellations, non-product charges and problematic numeric values enter sales measures.
  Use stock codes for grouping and disclose unresolved descriptions; do not assume a
  filled placeholder is a verified product name.
- **Customer purchasing behaviour:** conditionally suitable for customers with recorded IDs.
  The missing-ID share limits representativeness; unknown buyers cannot be linked reliably
  across invoices or treated as one customer. Retain their lines for appropriate aggregate
  transaction analysis and apply the privacy controls above to detailed customer work.
- **Future sales:** suitable only as a potential historical forecasting exercise after the
  pending numeric/outlier decisions and time-coverage validation. The record ends on
  {last_date_m4:%d %B %Y}; the final month is incomplete, the historical span is short, and
  one UK gift retailer with wholesale activity does not represent present-day retail overall.

**Remaining limitations:** Member 1's within-sheet duplicate removal is an assumption;
description variants and ambiguous locations remain flagged; ID missingness is unresolved;
geographic representation is concentrated ({uk_share_m4:.2f}% UK-labelled transaction lines);
and no protected demographic fairness assessment is possible from the available fields.
Member 3's treatment may change these counts, so rerun the Member 4 diagnostics on the
agreed Member 3 output when it is supplied, before final group submission.
"""
display(Markdown(conclusion_m4))

**Member 4 completion note.** Categories, PII, measured bias diagnostics, validation
and a qualified fitness conclusion are complete for the supplied Member 2 output.
Member 2's section has been placed before the later sections; their code and Member
1's code are preserved. Member 3's work remains visibly pending. After it arrives,
agree one sequential handoff, replace the Member 4 input with that agreed frame,
rerun all diagnostics, and reassess the conclusion before submitting the GitHub link.

**References for Member 4**

- Chen, D. (2012). *Online Retail II*. UCI Machine Learning Repository.
  https://doi.org/10.24432/C5CG6D. Dataset documentation and licence:
  https://archive.ics.uci.edu/dataset/502/online+retail+ii.
  Source information supports the retailer, historical dates and wholesale context;
  measured percentages come from the notebook, not from the documentation.
- Course material supplied by the group: *W5 Data Profiling and Visualization*,
  pages 11, 21-25, 41-46, 49-51 and 57 (before/after profiling, fitness for purpose,
  consistency, privacy and bias checks). Course PDFs are not republished here.